# Ridge Regression — Intuition

**Goal.** Build a mental picture of ridge regression *before* touching any math. Three questions, answered with pictures only:

1. What goes wrong with plain OLS when the design matrix is huge or near-singular?
2. What does "penalising large coefficients" actually *look* like?
3. How does the tuning parameter $\lambda$ move us between OLS and the constant-mean predictor?

No formulas here. The math lives in `02_mathematics.ipynb`.

**One-line preview.** Ridge regression = OLS with an *extra* term in the loss that punishes large $\theta$. It always has a unique solution (even when OLS does not), it always has smaller coefficient norms than OLS, and it almost always has lower test error than the unregularised fit.

**Prerequisites.** `02_polynomial_regression/01_intuition.ipynb` — the bias–variance picture and the Runge phenomenon. We will reuse the same noisy curve to make the overfit / shrink story concrete.

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The problem — OLS coefficients explode at high degree

Fit a polynomial of degree d to a small noisy dataset. As d grows, the *fitted curve* may stay reasonable in the middle, but the **coefficient magnitudes** swing wildly: $\pm$10^2, $\pm$10^4, even $\pm$10^6. Those huge coefficients are exactly what make the curve wiggle near the edges (the Runge phenomenon of `02_polynomial_regression/04_statistics.ipynb` §4) — and what makes the predictions on new data unreliable.

Concretely: same n = 30 noisy points from sin(2x), fit at degrees 1, 5, 10, 15. Print the largest absolute coefficient at each degree.

In [ ]:
def true_fn(t):
    return np.sin(2.0 * t)

n = 30
x = np.linspace(-1, 1, n)
y = true_fn(x) + rng.normal(0, 0.15, size=n)

for d in [1, 5, 10, 15]:
    coefs = np.polyfit(x, y, d)
    print(f"degree {d:>2}:  max |theta_j| = {np.max(np.abs(coefs)):>10.2f}")

**Reading.** At d = 15 the largest coefficient is two to three orders of magnitude bigger than the y-range of the data. This is a red flag: the model is using huge positive coefficients on some monomials and huge negative ones on others, with the two roughly cancelling on the training points but not on new points.

## 2. The fix — punish the size of $\theta$

Add a single new term to the loss: a penalty proportional to the *squared length* of the coefficient vector.

    $L_{\text{ridge}}(\theta)$  =  (1/n) ||$\Phi$ $\theta$ - y||^2  +  $\lambda$ $\cdot$ ||$\theta$||^2
                  ────────────────       ──────────
                     fit the data         keep $\theta$ small

The single new knob is $\lambda$ $\ge$ 0 ("$\lambda$"):

- **$\lambda$ = 0** — back to plain OLS. No penalty. Coefficients are free to grow.
- **$\lambda$ → $\infty$** — the penalty dominates. The optimum is $\theta$ $\approx$ 0 and the prediction is essentially the mean of y.
- **In between** — a compromise. The model fits the data, but only as far as the penalty allows.

Demo: same degree-15 polynomial, fit by OLS ($\lambda$ = 0) and by ridge at three values of $\lambda$. Plot the fitted curves *and* the size of the largest coefficient.

In [ ]:
def ridge_fit(Phi, y, lam):
    """Closed-form ridge: theta = (Phi^T Phi + n * lam * I)^-1 Phi^T y.

    The factor n keeps lam comparable to the MSE-based formulation (see
    02_mathematics §1.2). The intercept column is in Phi; we still penalise it
    here for simplicity — 05_hands_on_programming.ipynb does the standard
    'don't-penalise-intercept' bookkeeping properly.
    """
    n_, p_ = Phi.shape
    A = Phi.T @ Phi + n_ * lam * np.eye(p_)
    return np.linalg.solve(A, Phi.T @ y)

d = 15
# Build the degree-d Vandermonde, columns 1, x, x^2, ..., x^d.
Phi = np.vander(x, N=d + 1, increasing=True)

xs = np.linspace(-1.05, 1.05, 400)
Phi_eval = np.vander(xs, N=d + 1, increasing=True)

lams = [0.0, 1e-4, 1e-2, 1.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, lam in zip(axes, lams):
    theta = ridge_fit(Phi, y, lam)
    ax.plot(xs, true_fn(xs), color="black", ls="--", lw=1, label="true f")
    ax.scatter(x, y, alpha=0.6, edgecolor="k")
    ax.plot(xs, Phi_eval @ theta, color="crimson", lw=2)
    ax.set_xlabel("x")
    ax.set_title(f"λ = {lam:g}\nmax |θ| = {np.max(np.abs(theta)):.1f}")
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8)
axes[0].set_ylabel("y")
plt.suptitle(f"Ridge on a degree-{d} polynomial — more λ, smaller coefficients, smoother curve")
plt.tight_layout()
plt.show()

**Reading.**

- $\lambda$ = 0 (OLS) — wild edge oscillations, max |$\theta$| in the hundreds.
- $\lambda$ = $10^{-4}$ — a tiny amount of regularisation already kills the worst oscillations.
- $\lambda$ = 10⁻^2 — looks like a clean low-degree fit, even though the model still has d + 1 = 16 parameters.
- $\lambda$ = 1 — the penalty dominates; the curve is almost flat (close to the mean of y).

The model class is unchanged (degree-15 polynomial in all four panels) — *only the loss* has changed. Ridge does **not** change which functions are representable; it only changes which one we pick.

## 3. The regularisation path — every coefficient shrinks toward zero

Track each coefficient $\theta_{j}$ as $\lambda$ sweeps from very small to very large. The result is the **regularisation path** — one curve per coefficient. Two facts you can read straight off the plot:

1. All curves eventually flatten to zero on the right (large $\lambda$ → $\theta$ → 0).
2. None of them is exactly zero before infinity — ridge *shrinks* but never *eliminates*. (That is what makes the Lasso different — `04_lasso_regression/` next.)

In [ ]:
lams = np.logspace(-6, 3, 60)
thetas_path = np.array([ridge_fit(Phi, y, lam) for lam in lams])  # (n_lam, d+1)

fig, ax = plt.subplots(figsize=(7, 4.5))
for j in range(d + 1):
    ax.plot(lams, thetas_path[:, j], lw=1)
ax.set_xscale("log")
ax.set_xlabel("λ  (log scale)")
ax.set_ylabel("coefficient θⱼ")
ax.set_title("Ridge regularisation path on a degree-15 polynomial")
ax.axhline(0, color="black", lw=0.5)
ax.set_ylim(-30, 30)
plt.show()

**Reading.** Wild magnitudes on the left (small $\lambda$). All sixteen curves squeeze toward zero on the right. The interesting *middle* region — where the model is flexible enough to fit but stable enough to generalise — is exactly the $\lambda$ window we will pick via cross-validation in `04_statistics.ipynb`.

## 4. Train vs. test error as $\lambda$ sweeps

Same U-shape as in polynomial regression, but now along a *continuous* knob $\lambda$ instead of the discrete d. Hold d fixed at 15; sweep $\lambda$; plot train MSE vs. test MSE.

In [ ]:
# A bigger sample with held-out test set.
N = 80
x_all = rng.uniform(-1, 1, size=N)
y_all = true_fn(x_all) + rng.normal(0, 0.15, size=N)
perm = rng.permutation(N)
tr, te = perm[:40], perm[40:]
x_tr, y_tr = x_all[tr], y_all[tr]
x_te, y_te = x_all[te], y_all[te]
Phi_tr = np.vander(x_tr, N=d + 1, increasing=True)
Phi_te = np.vander(x_te, N=d + 1, increasing=True)

lams = np.logspace(-6, 2, 60)
mse_tr, mse_te = [], []
for lam in lams:
    theta = ridge_fit(Phi_tr, y_tr, lam)
    mse_tr.append(np.mean((y_tr - Phi_tr @ theta) ** 2))
    mse_te.append(np.mean((y_te - Phi_te @ theta) ** 2))
mse_tr, mse_te = np.array(mse_tr), np.array(mse_te)

lam_star = lams[np.argmin(mse_te)]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lams, mse_tr, "o-", color="steelblue", label="train MSE")
ax.plot(lams, mse_te, "o-", color="crimson",   label="test  MSE")
ax.axvline(lam_star, color="black", ls=":", label=f"best λ ≈ {lam_star:.1e}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("λ (log scale)")
ax.set_ylabel("MSE (log scale)")
ax.set_title("Same U-shape — but along a continuous knob λ")
ax.legend()
plt.show()

**Reading.**

- Far left ($\lambda$ tiny) — almost-OLS, overfit, test MSE bad.
- Middle — the sweet spot, where the model uses the data but is not enslaved to it.
- Far right ($\lambda$ huge) — under-fit, predictions collapse toward ȳ, test MSE bad again.

The bottom of the U on the right side is *softer* than on the left — over-regularising hurts less than under-regularising. That is one reason ridge has become the default "safe" linear regressor.

## Takeaway

- **What changes:** the loss gets a new term $\lambda$ $\cdot$ $\|\theta\|^2$. Nothing else — same features, same model class, same $\Phi$.
- **What you gain:** the coefficient magnitudes are controlled, so the fitted curve does not blow up at the edges. Ridge also has a unique solution even when OLS does not (see `02_mathematics.ipynb` §2.2).
- **What you trade away:** a little bit of bias (no longer unbiased — Gauss–Markov no longer applies). For most real problems the reduction in variance is worth it.
- **What you tune:** $\lambda$. Continuous, picked by cross-validation. Bigger $\lambda$ → smoother / smaller coefficients → more bias, less variance.

Next: `02_mathematics.ipynb` gives the closed-form solution, proves the design matrix becomes *always* invertible, derives the SVD shrinkage formula, and connects ridge to Bayesian MAP inference under a Gaussian prior.